In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_PATH = REPO_ROOT / "data" / "processed" / "dataset.csv"

dataset = pd.read_csv(DATASET_PATH, parse_dates = ["date_parsed"])
print(dataset.shape)
print(dataset.columns.tolist())
dataset.head()

(273, 27)
['ticker', 'quarter', 'date_parsed', 'word_count', 'sentence_count', 'avg_sentence_length', 'flesch_kincaid_grade', 'numeric_density', 'negative_count', 'uncertainty_count', 'litigious_count', 'forward_looking_count', 'word_count_zscore', 'sentence_count_zscore', 'avg_sentence_length_zscore', 'flesch_kincaid_grade_zscore', 'numeric_density_zscore', 'negative_count_zscore', 'uncertainty_count_zscore', 'litigious_count_zscore', 'forward_looking_count_zscore', 'excess_return_1d', 'excess_return_3d', 'excess_return_5d', 'y_1d', 'y_3d', 'y_5d']


,ticker,quarter,date_parsed,word_count,sentence_count,avg_sentence_length,flesch_kincaid_grade,numeric_density,negative_count,uncertainty_count,...,negative_count_zscore,uncertainty_count_zscore,litigious_count_zscore,forward_looking_count_zscore,excess_return_1d,excess_return_3d,excess_return_5d,y_1d,y_3d,y_5d
0,AAPL,2019-Q3,2019-07-30,8355,440,18.988636,10.239787,26.211849,18,30,...,-1.932399,-2.063308,-0.402183,1.347831,0.031345,0.004137,-0.013460,0,0,1
1,AAPL,2020-Q1,2020-01-28,8267,459,18.010893,10.009808,24.434499,27,38,...,-1.268959,-1.506463,-0.402183,1.347831,0.021758,-0.009963,-0.002987,0,1,0
2,AAPL,2020-Q2,2020-04-30,8278,442,18.728507,10.225606,16.791496,51,55,...,0.500212,-0.323169,-1.653419,1.609909,0.010374,0.027566,0.043472,0,0,0
3,AAPL,2020-Q3,2020-07-30,8109,468,17.326923,9.551080,21.211000,36,66,...,-0.605520,0.442493,1.474671,0.386877,0.096787,0.121258,0.152131,0,0,0
4,AAPL,2020-Q4,2020-10-29,8970,494,18.157895,9.559400,19.175028,40,49,...,-0.310658,-0.740802,0.849053,0.299518,-0.045593,-0.060651,-0.029227,1,1,1


In [2]:
print(f"Date range: {dataset['date_parsed'].min()} to {dataset['date_parsed'].max()}")
print(f"\nRows per year:")
print(dataset['date_parsed'].dt.year.value_counts().sort_index())
print(f"\nRows per quarter (first/last 5):")
quarters = dataset['date_parsed'].dt.to_period('Q').value_counts().sort_index()
print(quarters.head())
print("...")
print(quarters.tail())

Date range: 2019-06-25 00:00:00 to 2023-02-02 00:00:00

Rows per year:
date_parsed
2019     14
2020     20
2021    111
2022    112
2023     16
Name: count, dtype: int64

Rows per quarter (first/last 5):
date_parsed
2019Q2    1
2019Q3    8
2019Q4    5
2020Q1    7
2020Q2    6
Freq: Q-DEC, Name: count, dtype: int64
...
date_parsed
2022Q1    29
2022Q2    26
2022Q3    29
2022Q4    28
2023Q1    16
Freq: Q-DEC, Name: count, dtype: int64


In [6]:
# time based splitting

TRAIN_END = pd.Timestamp("2022-08-01")
VAL_END = pd.Timestamp("2022-10-01")

train_df = dataset[dataset["date_parsed"] < TRAIN_END].copy()
val_df = dataset[(dataset["date_parsed"] >= TRAIN_END) & (dataset["date_parsed"] < VAL_END)].copy()
test_df = dataset[dataset["date_parsed"] >= VAL_END].copy()

# sanity check
assert len(train_df) + len(val_df) + len(test_df) == 273
assert train_df["date_parsed"].max() < val_df["date_parsed"].min()
assert val_df["date_parsed"].max() < test_df["date_parsed"].min()

print(f"Train: {len(train_df)} rows  ({train_df['date_parsed'].min().date()} to {train_df['date_parsed'].max().date()})")
print(f"Val:   {len(val_df)} rows  ({val_df['date_parsed'].min().date()} to {val_df['date_parsed'].max().date()})")
print(f"Test:  {len(test_df)} rows  ({test_df['date_parsed'].min().date()} to {test_df['date_parsed'].max().date()})")

Train: 218 rows  (2019-06-25 to 2022-07-29)
Val:   11 rows  (2022-08-02 to 2022-09-22)
Test:  44 rows  (2022-10-14 to 2023-02-02)


In [7]:
# Cell 2b — Class balance per split (across all 3 windows)
print("Class balance (% of rows with y=1, i.e., 'high-risk'):\n")
print(f"{'Split':<8} {'y_1d':>8} {'y_3d':>8} {'y_5d':>8}")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    y1 = df["y_1d"].mean() * 100
    y3 = df["y_3d"].mean() * 100
    y5 = df["y_5d"].mean() * 100
    print(f"{name:<8} {y1:>7.1f}% {y3:>7.1f}% {y5:>7.1f}%")

Class balance (% of rows with y=1, i.e., 'high-risk'):

Split        y_1d     y_3d     y_5d
Train       50.0%    50.9%    50.0%
Val         36.4%    45.5%    36.4%
Test        52.3%    45.5%    52.3%


In [8]:
# Cell 2c — Ticker coverage per split
train_tickers = set(train_df["ticker"])
val_tickers = set(val_df["ticker"])
test_tickers = set(test_df["ticker"])

print(f"Tickers in train: {len(train_tickers)} / 30")
print(f"Tickers in val:   {len(val_tickers)} / 30")
print(f"Tickers in test:  {len(test_tickers)} / 30")

# Tickers missing from some splits
only_train = train_tickers - val_tickers - test_tickers
missing_from_train = (val_tickers | test_tickers) - train_tickers

if only_train:
    print(f"\nTickers ONLY in train (model never tested on these): {sorted(only_train)}")
if missing_from_train:
    print(f"\nTickers in val/test but NOT in train (model has no baseline): {sorted(missing_from_train)}")

Tickers in train: 30 / 30
Tickers in val:   11 / 30
Tickers in test:  28 / 30

Tickers ONLY in train (model never tested on these): ['ETSY', 'META']


In [10]:
# Cell 2d — Feature drift between train and test
RAW_FEATURES = [
    "word_count", "sentence_count", "avg_sentence_length",
    "flesch_kincaid_grade", "numeric_density",
    "negative_count", "uncertainty_count", "litigious_count",
    "forward_looking_count",
]

print("Feature mean drift (test mean − train mean), all 9 raw features:\n")

drifts = {}
for col in RAW_FEATURES:
    train_mean = train_df[col].mean()
    test_mean = test_df[col].mean()
    train_std = train_df[col].std()
    drift_in_stds = (test_mean - train_mean) / train_std if train_std > 0 else 0
    drifts[col] = drift_in_stds

# Sort by magnitude of drift (largest first)
sorted_drifts = sorted(drifts.items(), key=lambda x: abs(x[1]), reverse=True)

print(f"{'Feature':<25} {'Train mean':>12} {'Test mean':>12} {'Drift (stds)':>15}")
print("-" * 70)
for col, drift in sorted_drifts:
    train_mean = train_df[col].mean()
    test_mean = test_df[col].mean()
    print(f"{col:<25} {train_mean:>12.2f} {test_mean:>12.2f} {drift:>+15.2f}")

Feature mean drift (test mean − train mean), all 9 raw features:

Feature                     Train mean    Test mean    Drift (stds)
----------------------------------------------------------------------
word_count                     9660.90      9258.57           -0.23
avg_sentence_length              17.91        17.47           -0.21
litigious_count                   7.06         5.70           -0.18
flesch_kincaid_grade              9.91         9.73           -0.16
sentence_count                  543.87       533.39           -0.10
negative_count                   46.74        48.30           +0.07
forward_looking_count            78.20        79.50           +0.04
uncertainty_count                72.45        71.70           -0.03
numeric_density                  20.73        20.82           +0.01


In [11]:
# Defining X and y for train, val and test across all 3 windows

# independent variables
RAW_FEATURES = ["word_count", "sentence_count", "avg_sentence_length", 
                "flesch_kincaid_grade", "numeric_density", "negative_count", 
                "uncertainty_count", "litigious_count", "forward_looking_count"]
ZSCORE_FEATURES = [f"{c}_zscore" for c in RAW_FEATURES]
FEATURE_COLS = RAW_FEATURES + ZSCORE_FEATURES

# target variables 
TARGET_COLS = ["y_1d", "y_3d", "y_5d"]

X_train = train_df[FEATURE_COLS].copy()
X_val = val_df[FEATURE_COLS].copy()
X_test = test_df[FEATURE_COLS].copy()

y_train = {w: train_df[w].copy() for w in TARGET_COLS}
y_val = {w: val_df[w].copy() for w in TARGET_COLS}
y_test = {w: test_df[w].copy() for w in TARGET_COLS}

print("Feature matrix shapes:")
print(f" X_train: {X_train.shape}")
print(f" X_val: {X_val.shape}")
print(f" X_test: {X_test.shape}")

print("\nTarget vector shapes (same across all 3 windows):")
print(f" y_train: {y_train['y_5d'].shape}")
print(f" y_val: {y_val['y_5d'].shape}")
print(f" y_test: {y_test['y_5d'].shape}")

print(f"\nFeature count: {len(FEATURE_COLS)} ({len(RAW_FEATURES)} raw + {len(ZSCORE_FEATURES)} z-scored)")
print(f"Sample features: {FEATURE_COLS[:3]} ... {FEATURE_COLS[-3:]}")

# sanity check
assert X_train.shape == (218, 18)
assert X_val.shape == (11, 18)
assert X_test.shape == (44, 18)
assert not X_train.isnull().any().any(), "X_train has nulls!"
assert not X_val.isnull().any().any(), "X_val has nulls!"
assert not X_test.isnull().any().any(), "X_test has nulls!"
print("\n All shape and null checks passed")

Feature matrix shapes:
 X_train: (218, 18)
 X_val: (11, 18)
 X_test: (44, 18)

Target vector shapes (same across all 3 windows):
 y_train: (218,)
 y_val: (11,)
 y_test: (44,)

Feature count: 18 (9 raw + 9 z-scored)
Sample features: ['word_count', 'sentence_count', 'avg_sentence_length'] ... ['uncertainty_count_zscore', 'litigious_count_zscore', 'forward_looking_count_zscore']

 All shape and null checks passed


In [15]:
# Logistic regression for baseline

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# standardizing training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# training 1 log reg per window
baseline_results = {}

for window in TARGET_COLS:
    model = LogisticRegression(max_iter = 1000, random_state = 42)
    model.fit(X_train_scaled, y_train[window])

    train_pred = model.predict_proba(X_train_scaled)[:, 1]
    val_pred = model.predict_proba(X_val_scaled)[:, 1]

    train_auc = roc_auc_score(y_train[window], train_pred)
    val_auc = roc_auc_score(y_val[window], val_pred)

    baseline_results[window] = {"train_auc": train_auc,
                                "val_auc": val_auc,
                                "gap": train_auc - val_auc}
    
print("Logistic Regression Baseline\n")
print(f"{'Window': <8} {'Train AUC': >12} {'Val AUC': >12} {'Gap': >8}")
print("-" * 44)
for window, r in baseline_results.items():
    print(f"{window: <8} {r['train_auc']: >12.3f} {r['val_auc']: >12.3f} {r['gap']: >+8.3f}")

Logistic Regression Baseline

Window      Train AUC      Val AUC      Gap
--------------------------------------------
y_1d            0.693        0.679   +0.014
y_3d            0.676        0.533   +0.143
y_5d            0.685        0.429   +0.257


In [16]:
# Sanity check — what is the model actually predicting on val?
window = "y_5d"
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train[window])
val_pred = model.predict_proba(X_val_scaled)[:, 1]

print(f"Val predictions for {window}:")
for true_label, pred_prob in zip(y_val[window].values, val_pred):
    print(f"  True={true_label}, Predicted prob of class 1={pred_prob:.3f}")

Val predictions for y_5d:
  True=1, Predicted prob of class 1=0.519
  True=0, Predicted prob of class 1=0.355
  True=0, Predicted prob of class 1=0.782
  True=0, Predicted prob of class 1=0.606
  True=0, Predicted prob of class 1=0.456
  True=1, Predicted prob of class 1=0.466
  True=1, Predicted prob of class 1=0.324
  True=1, Predicted prob of class 1=0.567
  True=0, Predicted prob of class 1=0.394
  True=0, Predicted prob of class 1=0.406
  True=0, Predicted prob of class 1=0.711
